# 13 · Classifieurs — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch13_classifieurs/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 13.Q1 à 13.Q12, 13.R1 à 13.R3, 13.1 à 13.7, 13.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★★ | |
| A | 13.12 à 13.15 | prédire les frontières de cinq classifieurs, choisir k, KNeighborsClassifier dans mylearn.neighbors, kNN et malédiction de la dimension (*prochaine session de génération*) | | | |
| B | 13.16 à 13.18 | SVM linéaire et paramètre C, relever des cercles en 3D, noyau RBF réglé par validation croisée (*prochaine session de génération*) | | | |
| C | 13.19 à 13.26 | instabilité d'un arbre, impuretés, meilleur split et arbres dans mylearn.tree, une colonne qui triche, plot_tree et élagage, arbre de régression (*prochaine session de génération*) | | | |
| D | 13.27 à 13.30 | Naïve Bayes sur des blobs puis des lunes, GaussianNB et MultinomialNB dans mylearn.naive_bayes, phrases de Holmes en sac de mots et TF-IDF (*prochaine session de génération*) | | | |
| E | 13.31 et 13.32 | régression logistique de scikit-learn, puis la tienne dans mylearn.logistic (*prochaine session de génération*) | | | |
| F | 13.33 à 13.36 | associer des frontières, accélérer son kNN, comparatif mylearn contre scikit-learn, défi MNIST (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Calculer à la main un vote de k plus proches voisins, une marge, une impureté, un gain de split et un posterior de Naïve Bayes.
- Écrire et vérifier contre scikit-learn les classifieurs de ta librairie : `KNeighborsClassifier`, les arbres de `mylearn.tree` (prêts pour les ensembles du ch. 14), `GaussianNB`, `MultinomialNB` et `LogisticRegression`.
- Régler $k$, C, $\gamma$ et la profondeur d'un arbre par validation croisée, et limiter l'overfitting d'un arbre.
- Comparer les classifieurs (accuracy, temps, mémoire, explicabilité) et choisir une baseline.

**Rappel express.** Distances : $\sqrt{\sum_j (x_j - z_j)^2}$ et $\sum_j \lvert x_j - z_j \rvert$ ; distance à un hyperplan : $\lvert \mathbf{w} \cdot \mathbf{x} + b \rvert / \lVert \mathbf{w} \rVert$, largeur de la bande $2/\lVert \mathbf{w} \rVert$ ; noyau RBF : $e^{-\gamma \lVert \mathbf{x} - \mathbf{z} \rVert^2}$ ; Gini : $1 - \sum_c p_c^2$ ; entropie : $-\sum_c p_c \log_2 p_c$ ; gain : $I(\text{parent}) - \frac{n_G}{n} I(G) - \frac{n_D}{n} I(D)$ ; Naïve Bayes : $\log P(c) + \sum_j \log P(x_j \mid c)$, normalisé par logsumexp ; lissage : $\frac{N_{cj} + \alpha}{N_c + \alpha d}$ ; régression logistique : $\hat p = \sigma(\mathbf{w} \cdot \mathbf{x} + b)$, gradient $\frac1n \mathbf{X}^\top (\hat{\mathbf{p}} - \mathbf{y})$. En Python : `sklearn.neighbors.KNeighborsClassifier`, `sklearn.svm.SVC`, `sklearn.tree` (`DecisionTreeClassifier`, `plot_tree`, `export_text`), `sklearn.naive_bayes` (`GaussianNB`, `MultinomialNB`), `sklearn.linear_model.LogisticRegression`, `sklearn.model_selection.GridSearchCV`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 13.1 à 13.7 et 🧮 13.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une suite de codes : les lettres dans l'ordre (`"XYZ"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la preuve ∂ 13.8, la réflexion (🗣️ 13.10, ⚖️ 13.11) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np
from scipy.stats import norm


def gini(counts):
    p = np.asarray(counts, dtype=float) / np.sum(counts)
    return float(1 - np.sum(p ** 2))


def entropy_bits(probs):
    p = np.asarray(probs, dtype=float)
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))


def posterior_first(l_first, l_second):
    """P(first class | x) from two joint log-probabilities."""
    return float(1 / (1 + math.exp(l_second - l_first)))


# R1: three penguins (flipper in mm, mass in g) and the training standard deviations
A_R1, B_R1, C_R1 = np.array([190.0, 3700.0]), np.array([210.0, 3750.0]), np.array([192.0, 3800.0])
SD_R1 = np.array([14.0, 800.0])

# 13.1: seven training samples, the query (0, 0)
X_131 = np.array([[1, 1], [-1.5, 0], [0, 2], [2, -1.5], [-2, -2], [3, 0], [0, -3.5]])
Y_131 = np.array([1, 0, 0, 1, 2, 2, 2])
D_131 = np.linalg.norm(X_131, axis=1)          # euclidean distances to (0, 0)
M_131 = np.abs(X_131).sum(axis=1)              # manhattan distances
ORDER_131 = np.argsort(D_131, kind="stable")


def vote_131(k, weighted=False):
    """Class shares of the k nearest samples (euclidean), uniform or 1/d votes."""
    idx = ORDER_131[:k]
    w = 1 / D_131[idx] if weighted else np.ones(k)
    share = np.array([w[Y_131[idx] == c].sum() for c in range(3)])
    return share / share.sum()


# 13.3: ten flippers, Adelie (0) and Gentoo (1)
FL_133 = np.array([181, 186, 190, 193, 197, 210, 205, 215, 217, 228])
SP_133 = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1])
VALUES_133 = np.unique(FL_133)
THRESHOLDS_133 = (VALUES_133[:-1] + VALUES_133[1:]) / 2


def gain_133(s, impurity="gini"):
    left, right = SP_133[FL_133 <= s], SP_133[FL_133 > s]
    if impurity == "gini":
        f = lambda y: gini(np.bincount(y, minlength=2))
    else:
        f = lambda y: entropy_bits(np.bincount(y, minlength=2) / len(y))
    return f(SP_133) - len(left) / 10 * f(left) - len(right) / 10 * f(right)


BEST_133 = float(max(THRESHOLDS_133, key=gain_133))

# 13.4: six points, w = (0.5, 0.5), b = -2
P_134 = np.array([[3, 3], [5, 4], [4, 5], [1, 1], [0, 1], [2, 0]], dtype=float)
Y_134 = np.array([1, 1, 1, -1, -1, -1])
W_134, B_134 = np.array([0.5, 0.5]), -2.0
F_134 = P_134 @ W_134 + B_134
NORM_134 = float(np.linalg.norm(W_134))

# 13.5: x = (1, 2), z = (3, 1)
X_135, Z_135 = np.array([1.0, 2.0]), np.array([3.0, 1.0])
PHI = lambda v: np.array([v[0] ** 2, math.sqrt(2) * v[0] * v[1], v[1] ** 2])
K_135 = lambda a, b: float(a @ b) ** 2

# 13.6: letter counts e, h, q, w
EN_136, FR_136 = np.array([30, 12, 1, 7]), np.array([38, 4, 8, 0])
TH_EN = (EN_136 + 1) / (EN_136.sum() + 4)
TH_FR = (FR_136 + 1) / (FR_136.sum() + 4)
WHERE, QUE = np.array([2, 1, 0, 1]), np.array([1, 0, 1, 0])
L_EN = math.log(0.6) + WHERE @ np.log(TH_EN)
L_FR = math.log(0.4) + WHERE @ np.log(TH_FR)

# 13.7: Gaussian naive Bayes, flipper and bill depth
G_137 = np.array([210.0, 215.0, 220.0, 225.0])
LA_FL, LP_FL = norm.logpdf(205, 190, 6.5), norm.logpdf(205, 217, 6.5)
LA_BD, LP_BD = norm.logpdf(16.5, 18.3, 1.2), norm.logpdf(16.5, 15.0, 1.0)
JA_137 = math.log(0.55) + LA_FL + LA_BD
JP_137 = math.log(0.45) + LP_FL + LP_BD

# 13.9: MNIST, 60 000 training and 10 000 test images of 784 pixels
N_TRAIN, N_TEST, P_PIX = 60_000, 10_000, 784

**13.Q1 — Pourquoi essayer des classifieurs classiques d'abord ?**

In [3]:
wb.record("13.Q1a", True, mistakes={"relis le §13.1 : à quoi sert un score obtenu en quelques secondes, quand on envisage un modèle coûteux ?": False})
wb.record("13.Q1b", "B", mistakes={"sur 2 000 lignes, que t'apprend un modèle aussi coûteux, et que coûte-t-il ?": "A",
          "attendre d'autres données n'apprend rien sur celles qu'on a : que peut-on mesurer dès maintenant ?": "C",
          "un réglage fixé au hasard et aucune évaluation : comment saurais-tu ce que vaut le modèle ?": "D"})
wb.record("13.Q1c", False, mistakes={"quelque chose garantit-il qu'un modèle plus complexe fasse mieux sur des données de cette taille ?": True})
wb.record("13.Q1d", True, mistakes={"qu'apprends-tu si un kNN bat nettement une régression logistique sur les mêmes données ?": False})

WB_ANSWER {"hash": "1e9f945bb34eb81e1a4fdfa9a7fa4433b44545e1b3a91ed53368a47524e30673", "id": "13.Q1a", "kind": "bool", "mistakes": {"103c93d9c163e089dd9be95b7331f6f95a5b38c78267bf1e918563cbf30eacfb": "relis le §13.1 : à quoi sert un score obtenu en quelques secondes, quand on envisage un modèle coûteux ?"}}
📝 Ex 13.Q1a : réponse enregistrée (bool).
WB_ANSWER {"hash": "14b96a210d424e3f85cf8ffbbdc71068644b8cc2d3a6c538176ba944c42cf063", "id": "13.Q1b", "kind": "str", "mistakes": {"1be8d32bb50acbdc75aeb2973fdd06d38beaa4f037181e229f3ac6bb079a2eb8": "attendre d'autres données n'apprend rien sur celles qu'on a : que peut-on mesurer dès maintenant ?", "4160c62fb177c6bd6a0f2bdb388a97efbda81cf1905676a00ce4dee7417c96d0": "un réglage fixé au hasard et aucune évaluation : comment saurais-tu ce que vaut le modèle ?", "9c40bb3216f0be3f34c38c8d05517e3d1b1d4961c23c551df44fc6f22d6fb1a2": "sur 2 000 lignes, que t'apprend un modèle aussi coûteux, et que coûte-t-il ?"}}
📝 Ex 13.Q1b : réponse enregistrée (s

**13.Q2 — Paramétrique ou non paramétrique ?**

In [4]:
wb.record("13.Q2a", "PNPNPN", mistakes={"élément 6 (le SVM RBF) : relis la formule de prédiction du §13.4 ; de quoi est faite la somme, et ce nombre dépend-il de n ?": "PNPNPP",
          "élément 3 (Naïve Bayes) : combien de nombres garde-t-il, et ce nombre dépend-il de n ?": "PNNNPN",
          "élément 4 (l'arbre sans limite) : que devient son nombre de feuilles quand les données bruitées doublent ?": "PNPPPN",
          "élément 5 (le SVM linéaire) : que garde-t-il après l'entraînement, à part w et b ?": "PNPNNN",
          "élément 1 (la régression logistique) : combien de nombres garde-t-elle ?": "NNPNPN",
          "tu as inversé les deux familles : relis la définition du §13.2": "NPNPNP"})
wb.record("13.Q2b", "B", mistakes={"que garde ce modèle après l'entraînement : des poids, ou les exemples ?": "A",
          "que garde ce modèle après l'entraînement : des statistiques par classe, ou les exemples ?": "C",
          "que garde ce modèle après l'entraînement : un hyperplan, ou les exemples ?": "D"})
wb.record("13.Q2c", True, mistakes={"relis la définition d'un modèle paramétrique : son nombre de paramètres dépend-il de n ?": False})
wb.record("13.Q2d", False, mistakes={"un arbre ou un SVM à noyau n'apprend-il vraiment aucun nombre ? Relis la définition d'un classifieur non paramétrique (§13.2)": True})

WB_ANSWER {"hash": "210e380454201317b4e9c825702d13b431ae68875bb559a169c32adcd0ea29cf", "id": "13.Q2a", "kind": "str", "mistakes": {"2e01ba82b847891bf3cbe3cb22fa98e0f22f3a45b824a4e7ff1f41566f34db31": "tu as inversé les deux familles : relis la définition du §13.2", "5861d2139c9f56fb25248d3f4d4d821c0ee6b29b6124ba86ed0e2cbfc6c6a7cf": "élément 1 (la régression logistique) : combien de nombres garde-t-elle ?", "6e4ae1fdf6b47f70df5bfb95a2f1fde3f5de034b4bdddfaeb0c9b889946cfb30": "élément 6 (le SVM RBF) : relis la formule de prédiction du §13.4 ; de quoi est faite la somme, et ce nombre dépend-il de n ?", "7ae87479f06ba0874424c0745b68512cab6c60728c297c7802d0747fbab9a7bc": "élément 3 (Naïve Bayes) : combien de nombres garde-t-il, et ce nombre dépend-il de n ?", "be9abe75cb6e60b00e15b78d691b4b2fc55ee7b8e8d85d88898173d067fe0cc3": "élément 4 (l'arbre sans limite) : que devient son nombre de feuilles quand les données bruitées doublent ?", "c0789fec1fd66bd35295fba7fae9cd579890ab967ed79d9012731329bd

**13.Q3 — kNN n'est pas k-means**

In [5]:
wb.record("13.Q3a", "A", mistakes={"un algorithme non supervisé regarde-t-il les labels ?": "B",
          "l'un des deux est non supervisé : lequel ?": "C",
          "que fait voter le kNN ?": "D"})
wb.record("13.Q3b", "B", mistakes={"c'est le k du k-means ; relis le pseudo-code du kNN": "A",
          "le nombre de features est p ; relis le pseudo-code du kNN": "C",
          "le nombre de classes ne se choisit pas ; relis le pseudo-code du kNN": "D"})
wb.record("13.Q3c", True, mistakes={"relis ce que fait fit dans le pseudo-code du kNN : quel calcul y vois-tu ?": False})
wb.record("13.Q3d", True, mistakes={"un algorithme non supervisé connaît-il le nom des classes ?": False})

WB_ANSWER {"hash": "2e6e64a778af3da0608b5944e4669540f82c23fce7a4a97c32c27fbcb107f531", "id": "13.Q3a", "kind": "str", "mistakes": {"0cd7bf9d30d1ea9ec7ab84294378141253ad993c0bc018ec5a9d2f0cf84536fb": "un algorithme non supervisé regarde-t-il les labels ?", "ab114e0d9db325f7534a89d81777cdd56a948633cf88653a83be83d077859482": "que fait voter le kNN ?", "fc3be2ab5246f088797903eb1ff2c772ca03b8338d27d4a68954a8a10974a42d": "l'un des deux est non supervisé : lequel ?"}}
📝 Ex 13.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "a5c95267dd45ccb7baee17a234bc0394cc03956cf80cc02e7d8ccafb3c96f728", "id": "13.Q3b", "kind": "str", "mistakes": {"7228460f5cd261cca7090573478cd10167810e3a6dac9900481f4010f315c984": "c'est le k du k-means ; relis le pseudo-code du kNN", "73445aaa33bbb1d496659c1eb6344b5588f3320a1ca743dfabe924ec0ac69286": "le nombre de classes ne se choisit pas ; relis le pseudo-code du kNN", "9dfa7187203713b8696d3e604af015fd4caf12fe11f73cfdaddaf4fa8b9a19e2": "le nombre de features est p ; 

**13.Q4 — Petit k, grand k**

In [6]:
wb.record("13.Q4a", 1.0, decimals=2, mistakes={"sans doublon, quel est le plus proche voisin d'un point d'entraînement, cherché dans l'entraînement ?": 0.0,
          "une accuracy s'écrit ici entre 0 et 1, pas en pourcentage": 100.0,
          "sans doublon, sur quel point d'entraînement un 1-NN pourrait-il se tromper ?": 0.99})
wb.record("13.Q4b", "C", mistakes={"avec k = 1, un point du groupe a-t-il un voisin du groupe ? Qui gagne le vote ?": "A",
          "avec k = 3, combien de voisins d'un point du groupe peuvent venir du groupe, sur 6 points ?": "B"})
wb.record("13.Q4c", "B", mistakes={"avec k = n, combien d'exemples votent, et le résultat dépend-il encore du point ?": "A",
          "le vote est déterministe : qui vote quand k = n ?": "C",
          "k = n est permis : qui vote alors ?": "D"})
wb.record("13.Q4d", False, mistakes={"quelle valeur de k maximise toujours l'accuracy d'entraînement, et que dit-elle de la généralisation ?": True})
wb.record("13.Q4e", True, mistakes={"avec deux classes, un nombre impair de votes peut-il se partager en deux moitiés égales ?": False})

WB_ANSWER {"decimals": 2, "hash": "a22dd5d8d92adda8c72e37ab50e014b2dbced1bc325f178a32e28a98d2509da5", "hash_coarse": "fb55eda404970093091380d908724c2da8f8e0afea09bd11f8d91a1d61e39f71", "id": "13.Q4a", "kind": "float", "magnitude": 0, "mistakes": {"5f90ff26ef81ab9699630c9507281447e92963674b75d6370d211a00a3747e0f": "sans doublon, sur quel point d'entraînement un 1-NN pourrait-il se tromper ?", "96f2b7057ce157e6084b41bb416302554fd956e606d5b864905624e0a34cd4ff": "une accuracy s'écrit ici entre 0 et 1, pas en pourcentage", "c1fa708457600e7849d984ffeb705bce44fe063a4662760cbdba04812a68f761": "sans doublon, quel est le plus proche voisin d'un point d'entraînement, cherché dans l'entraînement ?"}, "sign": 1}
📝 Ex 13.Q4a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "8b8959ac0e1c55cf8997d3c6f7cc38b1131e14d0b0d1b1ac66bdbd96c69230fe", "id": "13.Q4b", "kind": "str", "mistakes": {"0d5472a60b1dc0b21263ba6c56e43dbbeca7286c6d31d2382ce3c5dfc519cc23": "avec k = 3, combien de voisins d'

**13.Q5 — Marge, vecteurs de support et C**

In [7]:
wb.record("13.Q5a", True, mistakes={"relis la définition des vecteurs de support : quels points fixent la solution ?": False})
wb.record("13.Q5b", "B", mistakes={"C multiplie les violations dans la loss : quand elles coûtent plus cher, l'optimisation en tolère-t-elle plus ?": "A",
          "C est un poids de la loss : peut-il ne rien changer ?": "C"})
wb.record("13.Q5c", "B", mistakes={"avec une marge souple, que deviennent les points dans la bande ou du mauvais côté ? Relis le §13.4": "A",
          "un point loin de la frontière, du bon côté, influence-t-il la solution ?": "C"})
wb.record("13.Q5d", True, mistakes={"le noyau RBF dépend d'une distance : une feature en grands nombres la dominerait-elle ?": False})

WB_ANSWER {"hash": "87f55b9f9021b27edfba7fdfae36894b3c909258ffd49efc10b510683e7eed7f", "id": "13.Q5a", "kind": "bool", "mistakes": {"2ca716b0e3f85e30cdd3b6e7f313e9524e7decca48447320bf19feb4e452eb28": "relis la définition des vecteurs de support : quels points fixent la solution ?"}}
📝 Ex 13.Q5a : réponse enregistrée (bool).
WB_ANSWER {"hash": "10d71ecf7e1b5bed078f7a2ea4f938c9e49496e2aaa0e54322fd34e19a1fc0f5", "id": "13.Q5b", "kind": "str", "mistakes": {"19656b54110d5884dcc294bb6f8a9f6b634b534cef75c76f11349a3f157ddec0": "C multiplie les violations dans la loss : quand elles coûtent plus cher, l'optimisation en tolère-t-elle plus ?", "b30376315c78ae82da7cd1d70eb5d64ceaace2c7091d231b6bceffe86c3cc4d6": "C est un poids de la loss : peut-il ne rien changer ?"}}
📝 Ex 13.Q5b : réponse enregistrée (str).
WB_ANSWER {"hash": "448eaf3b7a1a48f5ebcab3c4e5f22f4020fc6c73b6b7042f2f518a5f04d60768", "id": "13.Q5c", "kind": "str", "mistakes": {"8f46732ff7f2816a57eac94b0c3e6e54dbd336f04bd0b0d25add097245b1d

**13.Q6 — Ce que l'astuce du noyau évite de calculer**

In [8]:
wb.record("13.Q6a", "B", mistakes={"c'est justement ce que l'astuce évite de construire : relis l'encadré 🧮 sur l'astuce du noyau": "A",
          "le noyau calcule une ressemblance dans l'espace relevé, pas seulement dans l'espace de départ": "C",
          "le noyau ne choisit rien : relis l'encadré ⚠️ sur l'astuce du noyau": "D"})
wb.record("13.Q6b", 36, mistakes={"relis la définition du noyau polynomial : où est le degré 2 ?": 6})
wb.record("13.Q6c", 1, decimals=2, mistakes={"quelle est la distance d'un point à lui-même ? Remplace-la dans la formule": 0})
wb.record("13.Q6d", True, mistakes={"avec un grand γ, la ressemblance chute-t-elle vite ou lentement avec la distance ? Relis la figure du noyau RBF": False})
wb.record("13.Q6e", False, mistakes={"qui choisit le noyau, C et γ ? Relis l'encadré ⚠️ sur l'astuce du noyau": True})

WB_ANSWER {"hash": "6db7c4710be53e4ff2a6766c4353db3c11c6bfa1310a97857e4aeaac89093cc6", "id": "13.Q6a", "kind": "str", "mistakes": {"531657582ab585c5d3059be49b5f069c13f0a0c00f1d6d276108a1c465616eb3": "le noyau calcule une ressemblance dans l'espace relevé, pas seulement dans l'espace de départ", "79a1596b3a3de7d89e198ac6362026a0b59c987cf99115df19e362c354458173": "c'est justement ce que l'astuce évite de construire : relis l'encadré 🧮 sur l'astuce du noyau", "d6a620c47344af71ea1d666da958e78fd59fdf473dcf8259227d07a627e9c55f": "le noyau ne choisit rien : relis l'encadré ⚠️ sur l'astuce du noyau"}}
📝 Ex 13.Q6a : réponse enregistrée (str).
WB_ANSWER {"hash": "74db0ef590badad54e61540b33b572363d11b2a55e16c92c3c85dcf4b63f6494", "id": "13.Q6b", "kind": "int", "magnitude": 1, "mistakes": {"515a2914cf77cf318cc06a3f46479bc05c341b53161d303b8f37b38ccb2dcdc4": "relis la définition du noyau polynomial : où est le degré 2 ?"}, "sign": 1}
📝 Ex 13.Q6b : réponse enregistrée (int).
WB_ANSWER {"hash": "9f204

**13.Q7 — Racine, feuilles, profondeur : le vocabulaire des arbres**

In [9]:
wb.record("13.Q7a", 3, mistakes={"compte aussi la feuille de la branche « non » de la racine": 2,
          "une feuille n'a pas d'enfant : la racine et le second nœud n'en sont pas": 5})
wb.record("13.Q7b", 2, mistakes={"la profondeur compte les arêtes, pas les nœuds, du chemin le plus long": 3,
          "la profondeur se lit sur le chemin le plus long, pas le plus court": 1})
wb.record("13.Q7c", 2, mistakes={"un nœud interne pose une question ; les feuilles n'en posent pas": 5,
          "compte aussi la racine": 1})
wb.record("13.Q7d", 8, mistakes={"combien de feuilles au plus à la profondeur 1, puis 2 ? Que fait chaque niveau de plus ?": 6,
          "à la profondeur 3, combien de niveaux d'arêtes y a-t-il sous la racine ?": 16})
wb.record("13.Q7e", True, mistakes={"combien d'arêtes sépare la racine d'elle-même ?": False})

WB_ANSWER {"hash": "6177737b97bdcfc84a8a9eb9c7987c4796394f13a85f15f5bd232c38ac233cc8", "id": "13.Q7a", "kind": "int", "magnitude": 0, "mistakes": {"4c41b702780be1c2fb56489d2233eacd78787710dd6946d8c5daec29bf6c96e6": "compte aussi la feuille de la branche « non » de la racine", "54938419ab0bec2d0232148f5ec3fb93c4a8ad3e909dcb23ff2ff7c9a346685b": "une feuille n'a pas d'enfant : la racine et le second nœud n'en sont pas"}, "sign": 1}
📝 Ex 13.Q7a : réponse enregistrée (int).
WB_ANSWER {"hash": "bfc58fa2b4345bd411b5955e97ddd42eef5d8b18946c031a9effa1fa3d2a7c47", "id": "13.Q7b", "kind": "int", "magnitude": 0, "mistakes": {"069b5435ad85e6341353815abb80d6b41644536321749640a0ebdac77e549af9": "la profondeur compte les arêtes, pas les nœuds, du chemin le plus long", "8b2fdb357cd623ce1b9f8785aefcc211b619585268066a0b7156c96285a5dcc7": "la profondeur se lit sur le chemin le plus long, pas le plus court"}, "sign": 1}
📝 Ex 13.Q7b : réponse enregistrée (int).
WB_ANSWER {"hash": "aebb4c89276808695ce1813cd7

**13.Q8 — Pourquoi un arbre surapprend-il ?**

In [10]:
wb.record("13.Q8a", 1.0, decimals=2, mistakes={"un arbre sans limite s'arrête-t-il avant d'avoir des feuilles pures ?": 0.5,
          "une accuracy s'écrit ici entre 0 et 1, pas en pourcentage": 100.0,
          "sur quel exemple d'entraînement un arbre aux feuilles pures pourrait-il se tromper ?": 0.99})
wb.record("13.Q8b", False, mistakes={"une question « x_j ≤ s ? » dépend-elle de l'unité de x_j, ou seulement de l'ordre de ses valeurs ?": True})
wb.record("13.Q8c", "B", mistakes={"relis l'expérience du livre sur deux tirages de 70 % de données bruitées (§13.5.1)": "A",
          "si la première question change, que devient le reste de l'arbre ?": "C"})
wb.record("13.Q8d", "B", mistakes={"une question ne regarde qu'une feature : peut-elle tracer une droite oblique ?": "A",
          "une question ne regarde qu'une feature : peut-elle tracer une courbe ?": "C",
          "une seule droite, c'est un arbre de profondeur 1 ; et ensuite ?": "D"})

WB_ANSWER {"decimals": 2, "hash": "2bb54f1e8b878a7ec5caa2702276a27eb10a9c4e388e307ee5f674985910ff78", "hash_coarse": "910d0fd2b2b268af80a650d115b89912d6f73104c1f95030235904334813d596", "id": "13.Q8a", "kind": "float", "magnitude": 0, "mistakes": {"e1e15547fd944b61eff24731a87a5f25a2d18819030a7fcce3619ea207e67aa1": "sur quel exemple d'entraînement un arbre aux feuilles pures pourrait-il se tromper ?", "e65ca1fe393c5866dcd9f9957695e18ee8779e71f32a7210541be1c8d45ff541": "un arbre sans limite s'arrête-t-il avant d'avoir des feuilles pures ?", "f0b0ecf4214f2ddc2db3422836931f9b557c0691872a054c3c52ca763b602bb2": "une accuracy s'écrit ici entre 0 et 1, pas en pourcentage"}, "sign": 1}
📝 Ex 13.Q8a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "f339e94cd8012f1121b552d220bb3485c43578fc8c436ddada45237695f31c4c", "id": "13.Q8b", "kind": "bool", "mistakes": {"dd8938b620956205d96632a004f43838b9588ac9767790b7fbced1d1794b9295": "une question « x_j ≤ s ? » dépend-elle de l'unité de x_j

**13.Q9 — Gini ou gain d'information ?**

In [11]:
wb.record("13.Q9a", gini([1, 1]), decimals=2, mistakes={"relis la formule de Gini : 1 − Σ p²": 1})
wb.record("13.Q9b", entropy_bits([0.5, 0.5]), decimals=2, mistakes={"relis la formule de l'entropie : −Σ p log₂ p": 0.5, "en bits, le logarithme est en base 2": 0.69})
wb.record("13.Q9c", gini([1, 0]), decimals=2, mistakes={"dans un nœud pur, que vaut la proportion de sa classe, et donc Σ p² ?": 1})
wb.record("13.Q9d", "B", mistakes={"sans pondération, un enfant d'un seul exemple compterait autant que l'autre : relis la formule du gain": "A",
          "relis la formule du gain : quelle fraction multiplie I(gauche) ?": "C",
          "relis la formule du gain : le nombre de classes y apparaît-il ?": "D"})
wb.record("13.Q9e", True, mistakes={"relis le paragraphe « Gini ou entropie ? » et compare les courbes de la figure des impuretés": False})

WB_ANSWER {"decimals": 2, "hash": "be610bb8efea36f9f6755ee1a7973082d85571de27a0405b22edf365ff1f4614", "hash_coarse": "027e85f7eda66c45d2e82f225717159df68ce1f03b63cbbda383dd8940ff405e", "id": "13.Q9a", "kind": "float", "magnitude": -1, "mistakes": {"694395344a33b895ed9dc9c852ceff0687005edffade42783ae0295bbf0028e0": "relis la formule de Gini : 1 − Σ p²"}, "sign": 1}
📝 Ex 13.Q9a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "e15e8807e5a7f325cf5a749c98ab0e4eb9415e504ac6bfc197b53ca5a4bcf242", "hash_coarse": "1126feaae6da55344dfb4fe6f66449be46ca95181fe42fdfcedca2275b27d72e", "id": "13.Q9b", "kind": "float", "magnitude": 0, "mistakes": {"ecea5d3a4602baf1af49b0254ad5dc2f1d77759b350b8620b954de47f6b043f7": "relis la formule de l'entropie : −Σ p log₂ p", "eda07e1095c5cb460932d64f889ff16702c6c41145e45e3f9c4afa83e529a638": "en bits, le logarithme est en base 2"}, "sign": 1}
📝 Ex 13.Q9b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": 

**13.Q10 — Limiter la profondeur ou élaguer ?**

In [12]:
wb.record("13.Q10a", "PPEPE", mistakes={"élément 3 (ccp_alpha) : agit-il pendant la croissance, ou sur l'arbre complet ? Relis le §13.5.3": "PPPPE",
          "élément 5 : on retire des sous-arbres d'un arbre déjà construit ; pendant ou après la croissance ?": "PPEPP",
          "élément 2 (min_samples_leaf) : interdit-il une coupe pendant la croissance, ou retire-t-il une branche après ?": "PEEPE",
          "élément 4 (min_samples_split) : interdit-il une coupe pendant la croissance, ou retire-t-il une branche après ?": "PPEEE",
          "élément 1 (max_depth) : interdit-il une coupe pendant la croissance, ou retire-t-il une branche après ?": "EPEPE",
          "tu as inversé les deux familles : relis les deux paragraphes du §13.5.3": "EEPEP"})
wb.record("13.Q10b", True, mistakes={"dans R_α(T) = R(T) + α|T|, chaque feuille coûte α : quand α grandit, un arbre à beaucoup de feuilles devient-il plus intéressant ?": False})
wb.record("13.Q10c", "B", mistakes={"un réglage choisi sur le test rend le score de test optimiste (ch. 8)": "A",
          "le plus profond, c'est l'arbre sans limite : relis le §13.5.1": "C",
          "quelle profondeur maximise toujours l'accuracy d'entraînement ?": "D"})

WB_ANSWER {"hash": "db57df5da1d33c7500bf58b684a28ac3b261ed438de7f0d295c62b75e1aa0596", "id": "13.Q10a", "kind": "str", "mistakes": {"7b68b3db23489c010eca7ded3ee04a5b6d159058e809c49a13c5cb52199c8b33": "élément 2 (min_samples_leaf) : interdit-il une coupe pendant la croissance, ou retire-t-il une branche après ?", "8511d2e260c9e1ea45e358a31dbcbd6332185cd1283495c357c45a6c3a8a8fa0": "élément 3 (ccp_alpha) : agit-il pendant la croissance, ou sur l'arbre complet ? Relis le §13.5.3", "93eff01e43070901134812752b0941209b507afc7c751e38878f4ce6965f2f9e": "élément 1 (max_depth) : interdit-il une coupe pendant la croissance, ou retire-t-il une branche après ?", "acef144e61e8ef385b4376677ede5d6f81fca764c6bef01a261bc66b0807eb57": "élément 4 (min_samples_split) : interdit-il une coupe pendant la croissance, ou retire-t-il une branche après ?", "d5c6a6851fc633041823941b7b04cc7e1493e60a10174c7dfb82c49bd7e28b1d": "élément 5 : on retire des sous-arbres d'un arbre déjà construit ; pendant ou après la crois

**13.Q11 — Ce qui est vraiment « naïf » dans Naïve Bayes**

In [13]:
wb.record("13.Q11a", "B", mistakes={"c'est l'explication du livre ; relis l'encadré ⚠️ sur ce qui est « naïf »": "A",
          "la forme gaussienne est un choix de modèle, il y a d'autres variantes : quelle hypothèse leur est commune ?": "C",
          "les priors sont les fréquences des classes : quelle hypothèse permet le produit des P(x_j | c) ?": "D"})
wb.record("13.Q11b", True, mistakes={"d'où viennent les moyennes et les variances des gaussiennes ?": False})
wb.record("13.Q11c", 2 * 4 * 3, mistakes={"tu as compté une loi par classe et par feature : combien de nombres faut-il pour une loi normale ?": 12,
          "compte une loi par classe et par feature, pas par feature seulement": 8,
          "relis la question : quels nombres compte-t-on, et lesquels sont exclus ?": 27})
wb.record("13.Q11d", False, mistakes={"des features corrélées comptent plusieurs fois la même information : vers où cela pousse-t-il les probabilités ?": True})
wb.record("13.Q11e", True, mistakes={"que vaut le logarithme de 0, et un produit qui contient un facteur nul ?": False})

WB_ANSWER {"hash": "27430e86801b9439892af2230bfd7a5907fb4133c08208c55bd7a2191d3b8966", "id": "13.Q11a", "kind": "str", "mistakes": {"0af2ab3849cd4ff3740847181ab185fee9f1d445cad2776426752de1cdba1843": "la forme gaussienne est un choix de modèle, il y a d'autres variantes : quelle hypothèse leur est commune ?", "535e5d91e5614eb3e9454d4d8acde8fb13c43f1f9932e20676116ef64f80b18e": "les priors sont les fréquences des classes : quelle hypothèse permet le produit des P(x_j | c) ?", "6c7e1418e2c38f8868e5e186c1fbad1fa1486114b4c3a00ad5e9c5f25d7bcefc": "c'est l'explication du livre ; relis l'encadré ⚠️ sur ce qui est « naïf »"}}
📝 Ex 13.Q11a : réponse enregistrée (str).
WB_ANSWER {"hash": "c8abdff30a9d14a98fe7abdefd6049e0035de423cbfd5019cbe6c358d3b4bec7", "id": "13.Q11b", "kind": "bool", "mistakes": {"55f96ae593792c4077b16d90783bd232671d7c2a38365c7b8e6d96348a66dbcf": "d'où viennent les moyennes et les variances des gaussiennes ?"}}
📝 Ex 13.Q11b : réponse enregistrée (bool).
WB_ANSWER {"hash": "cf9

**13.Q12 — Quel classifieur pour quelle situation ?**

In [14]:
wb.record("13.Q12a", "ANKLS", mistakes={"situations 3 et 5 : lequel intègre un exemple sans réentraînement, et lequel a deux hyperparamètres à régler pour une frontière non linéaire ?": "ANSLK",
          "situations 1 et 4 : lequel donne des règles « si … alors … », et lequel un poids par feature et des probabilités fiables ?": "LNKAS",
          "situations 2 et 4 : lequel est la baseline de quelques secondes sur des comptes de mots ?": "ALKNS",
          "situations 4 et 5 : lequel donne un poids par feature et des probabilités fiables, et lequel a deux hyperparamètres pour une frontière non linéaire ?": "ANKSL"})
wb.record("13.Q12b", True, mistakes={"avec des folds différents, l'écart entre deux scores mesure-t-il seulement la différence entre les modèles ?": False})

WB_ANSWER {"hash": "50f524d03a3e17107521e7505b7413a22e15dda68c763233897d3c380f0b5866", "id": "13.Q12a", "kind": "str", "mistakes": {"487a06a99f6592bbfe0dce21e0562fd0d5325ce7dec754c53cf27e890665f096": "situations 2 et 4 : lequel est la baseline de quelques secondes sur des comptes de mots ?", "65f52e4f04a9896f01b42cdca7a412ec84fcd3a4bea4b6a74069135ff9ed7ef0": "situations 3 et 5 : lequel intègre un exemple sans réentraînement, et lequel a deux hyperparamètres à régler pour une frontière non linéaire ?", "a0b8a2b4c943ac6ff954e2991ba52df131b010c161b5f73e8af3571a4dc6a123": "situations 1 et 4 : lequel donne des règles « si … alors … », et lequel un poids par feature et des probabilités fiables ?", "c22c9306b68cb31779ff1ba5d67506bbf2e0c9fce0401e4fd2a98be5443d983e": "situations 4 et 5 : lequel donne un poids par feature et des probabilités fiables, et lequel a deux hyperparamètres pour une frontière non linéaire ?"}}
📝 Ex 13.Q12a : réponse enregistrée (str).
WB_ANSWER {"hash": "21ca572edfe87b3

**13.R1 — Pourquoi standardiser avant de mesurer des distances**

In [15]:
wb.record("13.R1a", float(np.linalg.norm(C_R1 - A_R1)), decimals=2, mistakes={"tu as la distance au carré : prends sa racine": 10004,
          "c'est la distance de Manhattan ; l'euclidienne prend la racine de la somme des carrés": 102})
wb.record("13.R1b", float(np.linalg.norm(C_R1 - B_R1)), decimals=2, mistakes={"tu as la distance au carré : prends sa racine": 2824,
          "c'est la distance de Manhattan ; l'euclidienne prend la racine de la somme des carrés": 68})
wb.record("13.R1c", "B", mistakes={"compare les deux distances brutes que tu viens de calculer": "A"})
wb.record("13.R1d", float(np.linalg.norm((C_R1 - A_R1) / SD_R1)), decimals=3, mistakes={"tu as la distance au carré : prends sa racine": 0.036,
          "divise chaque différence par l'écart-type de sa propre feature": 7.143,
          "vérifie quel manchot tu as pris : la question porte sur C et A": 1.287})
wb.record("13.R1e", "A", mistakes={"calcule aussi la distance standardisée entre C et B, puis compare": "B"})
wb.record("13.R1f", True, mistakes={"relis la règle d'or du ch. 12 : où s'apprennent les paramètres d'un scaler ?": False})

WB_ANSWER {"decimals": 2, "hash": "892115a5418398dee52607d593e513a37d974aabd580b7cb7686ed8e2f815d62", "hash_coarse": "2bfaccd4b02aca3f4ae7d34b5d78cc806293c8f8d86cea348860fe6e1a135c14", "id": "13.R1a", "kind": "float", "magnitude": 2, "mistakes": {"8babb5e788c7d5639f27659f7e826ffc322b40338219dbf7735f71195e17ab2e": "tu as la distance au carré : prends sa racine", "db26a5a0f9a1bfa98b28de8b98dd5b469b475dc7c8c80cef8c9574850dcc2c74": "c'est la distance de Manhattan ; l'euclidienne prend la racine de la somme des carrés"}, "sign": 1}
📝 Ex 13.R1a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "ff4b386a6c62b4cd8a9bc41eff16b2888a166542b2f90bb2649db01c09e26632", "hash_coarse": "6394579e0ea4d4e5e818eda71b48e2394bc99e050e5d534255fa2da66977a61f", "id": "13.R1b", "kind": "float", "magnitude": 1, "mistakes": {"5f37bca6a37ce977712638b18e4f7388351f316808844bee08fb34d667f06dbf": "c'est la distance de Manhattan ; l'euclidienne prend la racine de la somme des carrés", "9135

**13.R2 — L'équation de la frontière d'un perceptron**

In [16]:
wb.record("13.R2a", 1, mistakes={"calcule w·x + b pour cette entrée : quel est son signe, et que sort alors le perceptron ?": 0, "la sortie d'un perceptron vaut 0 ou 1": 2})
wb.record("13.R2b", 0, mistakes={"calcule w·x + b pour cette entrée : quel est son signe, et que sort alors le perceptron ?": 1, "la sortie d'un perceptron vaut 0 ou 1": -1})
wb.record("13.R2c", "A", mistakes={"le OU sort 1 dès qu'une entrée vaut 1 : est-ce le cas pour (0, 1) ?": "B",
          "le OU exclusif n'est pas linéairement séparable (ch. 10) : un perceptron peut-il le réaliser ?": "C",
          "le NON-ET sort 1 pour (0, 0) : est-ce le cas ici ?": "D"})
wb.record("13.R2d", -1, decimals=2, mistakes={"isole x₂ dans x₁ + x₂ − 1,5 = 0 : quel coefficient multiplie x₁ ?": 1})
wb.record("13.R2e", 1.5, decimals=2, mistakes={"isole x₂ dans x₁ + x₂ − 1,5 = 0 : quel est le signe de l'ordonnée à l'origine ?": -1.5})
wb.record("13.R2f", 0.5 / math.sqrt(2), decimals=3, mistakes={"as-tu divisé par la norme de w ?": 0.5, "recalcule la norme de (1, 1)": 0.25})
wb.record("13.R2g", True, mistakes={"pour quelle valeur de z a-t-on σ(z) = 0,5 ?": False})

WB_ANSWER {"hash": "0537b73aef82eb8f922e222b3ae334345f1c595cebf0d7e4f692f8213470d9c0", "id": "13.R2a", "kind": "int", "magnitude": 0, "mistakes": {"6cf31472e76131e8928416a0beb41f9704ec1affaa033266fcde2ed333581212": "calcule w·x + b pour cette entrée : quel est son signe, et que sort alors le perceptron ?", "c729561bc7f348c7013afabf48e5577569b4fb48d4074c2aa56b1894cf8faa76": "la sortie d'un perceptron vaut 0 ou 1"}, "sign": 1}
📝 Ex 13.R2a : réponse enregistrée (int).
WB_ANSWER {"hash": "6d7a2934e976dbc27ed7e8c2f07935f5434f432ecb0d864bfa785aa1167f7eee", "id": "13.R2b", "kind": "int", "magnitude": null, "mistakes": {"5ceda18211149ff089d9d79c4937aae4d4e190f0a23fbd5e899bc44ca39714b1": "calcule w·x + b pour cette entrée : quel est son signe, et que sort alors le perceptron ?", "93782f72b1f6da0b6b972b0825974c4484dae41faf5b54d5395a2cad71df2627": "la sortie d'un perceptron vaut 0 ou 1"}, "sign": 0}
📝 Ex 13.R2b : réponse enregistrée (int).
WB_ANSWER {"hash": "70fe62171128e566fefdc79c0d74720a5cd87

**13.R3 — Entropie de deux distributions**

In [17]:
wb.record("13.R3a", entropy_bits([0.5, 0.25, 0.25]), decimals=2, mistakes={"en bits, log₂ 0,25 vaut −2 : vérifie ce terme": 1, "le logarithme est en base 2, pas népérien": 1.04})
wb.record("13.R3b", entropy_bits([0.8, 0.2]), decimals=3, mistakes={"le logarithme est en base 2, pas népérien": 0.5, "le signe moins rend l'entropie positive": -0.722})
wb.record("13.R3c", 2.0, decimals=2, mistakes={"c'est le nombre de valeurs ; l'entropie maximale est son logarithme en base 2": 4.0, "en bits, le logarithme est en base 2": 1.39})
wb.record("13.R3d", float(-0.5 * math.log2(0.8) - 0.5 * math.log2(0.2)), decimals=3, mistakes={"dans H(p, q), quels nombres sont les poids, et lesquels sont dans le log ?": 0.722,
          "dans H(p, q), les logarithmes sont-ils ceux des p_i ou des q_i ?": 1,
          "le logarithme est en base 2, pas népérien": 0.916})
wb.record("13.R3e", True, mistakes={"que vaut H(p, q) − H(p), et quel est son signe (ch. 6) ?": False})

WB_ANSWER {"decimals": 2, "hash": "ebb94f4f89face062ebca9973a15bab18c7aa18ff4fa918ef997db3894531471", "hash_coarse": "c8d85567a4e31a9ae36a3e85a326ee5fe5bdb3e968acdc30c8dad7fa875f44ae", "id": "13.R3a", "kind": "float", "magnitude": 0, "mistakes": {"5db5510d50695f4f4363d98f3b8ffeb4d7ff11d06d85b7caa6fd169bd2ae2459": "en bits, log₂ 0,25 vaut −2 : vérifie ce terme", "c348e5e8bf5e155cea9cc3ea531fb3e63818eda046cb6aa2f942f8167308a1ed": "le logarithme est en base 2, pas népérien"}, "sign": 1}
📝 Ex 13.R3a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "e1864ac4c7807b5a17ea6d9978a1774ecc7451c2038085585a36398256f2e508", "hash_coarse": "f00e537b393348b451f3def9db4d3570b4ddb7525a1342068e08d9bb6c756f4d", "id": "13.R3b", "kind": "float", "magnitude": -1, "mistakes": {"168e4d0f9bbf34335dd55b803a54b3d5c75e52faeb13516910603c95953c7d53": "le signe moins rend l'entropie positive", "c2e523a672927b1acd4c8e121c5bc36c3e3f75bfc40b6a1f044d878d564fa751": "le logarithme est en base

**Ex 13.1 — Vote des k plus proches voisins à la main**

In [18]:
wb.record("13.1a", int(ORDER_131[0]), mistakes={"as-tu calculé les distances euclidiennes, et numéroté les exemples à partir de 0 ?": 1})
wb.record("13.1b", int(Y_131[ORDER_131[0]]), mistakes={"refais le calcul en distance euclidienne": 0, "lis la classe du plus proche voisin dans le tableau": 2})
wb.record("13.1c", int(np.argmax(vote_131(3))), mistakes={"compte les classes des trois plus proches, pas seulement celle du premier": 1})
wb.record("13.1d", float(vote_131(3)[0]), decimals=3, mistakes={"c'est la part de l'autre classe représentée parmi les trois voisins": 0.333})
wb.record("13.1e", int(np.argmax(vote_131(5))), mistakes={"compte les votes des cinq plus proches : y a-t-il une égalité, et quelle classe la gagne dans scikit-learn ?": 1,
          "la classe 2 n'a qu'un vote parmi les cinq plus proches : compte à nouveau": 2})
wb.record("13.1f", int(np.argmax(vote_131(7))), mistakes={"avec k = 7, tous les exemples votent : compte les classes de tout l'entraînement": 0,
          "avec k = 7, tous les exemples votent : compte à nouveau les classes de tout l'entraînement": 1})
wb.record("13.1g", float(vote_131(3, weighted=True)[1]), decimals=3, mistakes={"c'est la part des votes uniformes ; pondère chaque voisin par 1/d": 0.333,
          "divise par la somme des poids, pas par k": 0.236,
          "tu as la part de l'autre classe": 0.623})
wb.record("13.1h", int(np.argmax(vote_131(7, weighted=True))), mistakes={"as-tu pondéré chaque vote par 1/d, puis additionné les poids de chaque classe ?": 2,
          "additionne aussi le poids du second exemple de cette classe, puis compare les sommes": 1})
wb.record("13.1i", [int(np.argsort(M_131, kind='stable')[0]), int(Y_131[np.argsort(M_131, kind='stable')[0]])], mistakes={"en Manhattan, additionne |x₁| + |x₂| pour chaque exemple": [0, 1]})

WB_ANSWER {"hash": "941d3e59e093b2bb561e4369cc73ce1a2b5e6258910319e27a3065ce97783e86", "id": "13.1a", "kind": "int", "magnitude": null, "mistakes": {"7a6a8bbcae0b01ab0d9d9883b125b8d57451c7ec389a40f17fd816d424aeea95": "as-tu calculé les distances euclidiennes, et numéroté les exemples à partir de 0 ?"}, "sign": 0}
📝 Ex 13.1a : réponse enregistrée (int).
WB_ANSWER {"hash": "fbaf78b50153eeb12a1ac4f7501e080646c74e767b09eed2537e536523af8e2a", "id": "13.1b", "kind": "int", "magnitude": 0, "mistakes": {"15b4fa9001c7aeb0880d0c8a35dddca93077525ef74149b6e6aebbeeec6bdc2e": "refais le calcul en distance euclidienne", "503eaa255e4680d02c648b0dd55144a72eec6c7b16ccde0e0e34144b19041e1b": "lis la classe du plus proche voisin dans le tableau"}, "sign": 1}
📝 Ex 13.1b : réponse enregistrée (int).
WB_ANSWER {"hash": "666dd2873cc9990435d7eb1f4e4deeaf748d95cea2f0a315ad9b8dbd496ffcb1", "id": "13.1c", "kind": "int", "magnitude": null, "mistakes": {"c226085351935795231d73519a4625ce0fbf57878e5412a57ccb988ea35fad

**Ex 13.2 — Gini et entropie d'une feuille**

In [19]:
wb.record("13.2a", gini([10, 90]), decimals=2, mistakes={"c'est la somme des carrés ; Gini, c'est 1 moins cette somme": 0.82,
          "relis la formule de Gini : 1 − Σ p²": 0.1})
wb.record("13.2b", entropy_bits([0.1, 0.9]), decimals=3, mistakes={"le logarithme est en base 2, pas népérien": 0.325})
wb.record("13.2c", 0.1, decimals=2, mistakes={"en répondant toujours B, sur quels exemples se trompe-t-on ?": 0.9})
wb.record("13.2d", 0.9, decimals=2, mistakes={"en répondant toujours A, sur quels exemples se trompe-t-on ? (Le livre se trompe ici, p. 527.)": 0.1})
wb.record("13.2e", 0.1 * 0.9 + 0.9 * 0.1, decimals=3, mistakes={"on se trompe de deux façons (vrai A et tiré B, vrai B et tiré A) : additionne-les": 0.09,
          "c'est la probabilité de tomber juste ; on demande celle de se tromper": 0.82})
wb.record("13.2f", gini([50, 30, 20]), decimals=2, mistakes={"c'est la somme des carrés ; Gini, c'est 1 moins cette somme": 0.38})
wb.record("13.2g", entropy_bits([0.5, 0.3, 0.2]), decimals=3, mistakes={"le logarithme est en base 2, pas népérien": 1.03})
wb.record("13.2h", gini([1, 1, 1]), decimals=3, mistakes={"c'est le maximum à deux classes ; que vaut chaque proportion avec trois classes à égalité ?": 0.5})
wb.record("13.2i", math.log2(3), decimals=3, mistakes={"le logarithme est en base 2, pas népérien": 1.099, "c'est le maximum à deux classes": 1})

WB_ANSWER {"decimals": 2, "hash": "e6103997af9464b5d23461a572ad56a7c92b13724dba6ff16624acff362a5dfd", "hash_coarse": "3bd346be3ef6a7c2afe097783c6397bba9f07a303dd1a4279080801554e50445", "id": "13.2a", "kind": "float", "magnitude": -1, "mistakes": {"8e25d97c996f7d6b189d7cf8e12fe4320ed1fa70bbef6125bcd84cf8297e08b0": "c'est la somme des carrés ; Gini, c'est 1 moins cette somme", "b8d2cec00b82c49b96e5ae160d9a3176fe277bb72bf54816ca0a245009151307": "relis la formule de Gini : 1 − Σ p²"}, "sign": 1}
📝 Ex 13.2a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "a7e9842c87b03265b4580dad8d5f031af3147aaa7042f4a99415884d5d63ef64", "hash_coarse": "62a94867d9a673e33d8c3122962f7dbd345d3b67798ad9195efba714fbe22314", "id": "13.2b", "kind": "float", "magnitude": -1, "mistakes": {"74ed1ff7eef1d18cf29d2967424c872a792cb039b3025f27ea97ec9613601766": "le logarithme est en base 2, pas népérien"}, "sign": 1}
📝 Ex 13.2b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decim

**Ex 13.3 — Choisir le meilleur seuil sur la longueur des nageoires**

In [20]:
wb.record("13.3a", len(THRESHOLDS_133), mistakes={"compte les intervalles entre valeurs consécutives, pas les valeurs": 10})
wb.record("13.3b", gini([6, 4]), decimals=2, mistakes={"c'est la somme des carrés ; Gini, c'est 1 moins cette somme": 0.52})
wb.record("13.3c", gain_133(201), decimals=2, mistakes={"pondère l'impureté de chaque enfant par sa part des exemples (5 sur 10)": 0.16})
wb.record("13.3d", gain_133(212.5), decimals=3, mistakes={"pondère l'impureté de chaque enfant par sa part des exemples": 0.235})
wb.record("13.3e", gain_133(207.5), decimals=3, mistakes={"pondère l'impureté de chaque enfant par sa part des exemples": -0.173})
wb.record("13.3f", BEST_133, decimals=1, mistakes={"calcule les gains au lieu de prendre le seuil « au milieu » des deux espèces": 207.5,
          "as-tu calculé le gain de tous les seuils candidats ?": 212.5})
wb.record("13.3g", gain_133(201, impurity='entropy'), decimals=3, mistakes={"pondère l'entropie de chaque enfant par sa part des exemples": 0.249,
          "le logarithme est en base 2, pas népérien": 0.423})
wb.record("13.3h", 1, mistakes={"chaque feuille prédit sa classe majoritaire : à droite du seuil, combien de manchots ne l'ont pas ?": 0})
wb.record("13.3i", 1, mistakes={"seul compte le côté du seuil : 203 est-il au plus égal au seuil ?": 0, "réponds 0 (Adélie) ou 1 (Papou)": 2, "réponds 0 (Adélie) ou 1 (Papou) ; quelle classe prédit la feuille où tombe 203 mm ?": -1})
wb.record("13.3j", int(sum(5 <= (FL_133 <= s).sum() <= 5 for s in THRESHOLDS_133)), mistakes={"chaque enfant doit recevoir au moins 5 manchots : sur 10, quel découpage reste possible ?": 0,
          "vérifie aussi l'enfant de droite": 5})

WB_ANSWER {"hash": "30fea9a6e81fa5a12d125b408ec327dbffd428caab5bd419cbafe2f12e5f61dd", "id": "13.3a", "kind": "int", "magnitude": 0, "mistakes": {"6136621a7c87534c416eb7878eab4fc8eeef2d11c59f400a8776bab4029c08fc": "compte les intervalles entre valeurs consécutives, pas les valeurs"}, "sign": 1}
📝 Ex 13.3a : réponse enregistrée (int).
WB_ANSWER {"decimals": 2, "hash": "60b82d83dc4782565628c93ca58bfbd5ef08243db11fd87f264097d324f84230", "hash_coarse": "0a933049a91e5c8c94d76ed163e00ec6b63b6604d121577b3d423102170cdb74", "id": "13.3b", "kind": "float", "magnitude": -1, "mistakes": {"e9b06e409cdbbb9422aed0e66aafbed080c317c4a0367eda49aa14e99afd3b6f": "c'est la somme des carrés ; Gini, c'est 1 moins cette somme"}, "sign": 1}
📝 Ex 13.3b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "b7cd3639763ca09bfd01bf191ce79fc769928b1654b8ec1830290879296a4f51", "hash_coarse": "278684f3eb2a8f367626ce3f2cd4a5eebbb74ec2f6fe342edea621ccc905eca0", "id": "13.3c", "kind": "float", 

**Ex 13.4 — Distance à une droite et marge d'un SVM**

In [21]:
wb.record("13.4a", NORM_134, decimals=4, mistakes={"c'est la norme au carré : prends sa racine": 0.5, "c'est la norme de Manhattan ; prends la racine de la somme des carrés": 1})
wb.record("13.4b", float(F_134[1]), decimals=2, mistakes={"n'oublie pas b = −2": 4.5})
wb.record("13.4c", float(abs(F_134[1]) / NORM_134), decimals=3, mistakes={"as-tu divisé par la norme de w ?": 2.5, "divise par la norme de w, pas par son carré": 5})
wb.record("13.4d", float(np.min(Y_134 * F_134)), decimals=2, mistakes={"multiplie f par y : pour les points de la classe −1, le signe change": -1.5})
wb.record("13.4e", [int(i) for i in np.flatnonzero(np.isclose(Y_134 * F_134, 1))], mistakes={"le point 4 vérifie y f = 1,5 : est-il sur le bord de la bande ?": [0, 3, 4, 5],
          "calcule y f pour les points de la classe −1 aussi": [0]})
wb.record("13.4f", 2 / NORM_134, decimals=3, mistakes={"c'est la demi-largeur 1/‖w‖ ; la bande va de f = −1 à f = +1": 1.414})
wb.record("13.4g", 1, mistakes={"calcule f(2, 3) et regarde son signe": -1, "réponds +1 ou −1 : le signe de f": 0})
wb.record("13.4h", True, mistakes={"compare |f(2, 3)| à 1": False})
wb.record("13.4i", False, mistakes={"le numérateur |f| et le dénominateur ‖w‖ sont multipliés par le même nombre : que devient le rapport ?": True})
wb.record("13.4j", True, mistakes={"le point 1 est-il sur le bord de la bande ?": False})

WB_ANSWER {"decimals": 4, "hash": "451676270a6928a981e2640d2d7828403f401bffe599697aef4bda08f0ce7e60", "hash_coarse": "ebedb77f6271b2833a90ef642c2caf80c5b6f5961a04111f46e2d5f6bbc6d701", "id": "13.4a", "kind": "float", "magnitude": -1, "mistakes": {"4453dd30399c7cf13229529c1b4117936f82b81dada19cea374229c4c8897d01": "c'est la norme au carré : prends sa racine", "56ad4171cb2f021d1f09c07372dd27bf40ba82a3905b973ce154ce9fc800fe58": "c'est la norme de Manhattan ; prends la racine de la somme des carrés"}, "sign": 1}
📝 Ex 13.4a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "3965f9b0d586d329c3636a468fa569d9351f0e6639ea2be1188284edef6213be", "hash_coarse": "66aa6f3a7db4eeac8a67c8f09c283e87af0ce366eb8bcd64dd05a26260f4b532", "id": "13.4b", "kind": "float", "magnitude": 0, "mistakes": {"0245b119aa24e95893635517dc13b94535d560370b2d8a1bd0e9b5bd37fa9dba": "n'oublie pas b = −2"}, "sign": 1}
📝 Ex 13.4b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 

**Ex 13.5 — L'astuce du noyau vérifiée à la main**

In [22]:
wb.record("13.5a", float(X_135 @ Z_135), decimals=2, mistakes={"multiplie les coordonnées de même rang : la première avec la première, la seconde avec la seconde": 7})
wb.record("13.5b", K_135(X_135, Z_135), decimals=2, mistakes={"relis la définition du noyau polynomial : où est le degré 2 ?": 5})
wb.record("13.5c", float(PHI(X_135)[1]), decimals=3, mistakes={"n'oublie pas le facteur √2": 2})
wb.record("13.5d", float(PHI(X_135) @ PHI(Z_135)), decimals=2, mistakes={"n'oublie pas le facteur √2 dans la deuxième coordonnée de chaque vecteur": 19})
wb.record("13.5e", K_135(X_135, X_135) - 2 * K_135(X_135, Z_135) + K_135(Z_135, Z_135), decimals=2, mistakes={"c'est la distance au carré dans l'espace de départ ; on la demande dans l'espace relevé": 5,
          "n'oublie pas le facteur 2 devant K(x, z)": 100})
wb.record("13.5f", math.exp(-0.5 * float((X_135 - Z_135) @ (X_135 - Z_135))), decimals=4, mistakes={"prends le carré de la distance dans l'exposant": 0.3269, "prends le carré de la distance dans l'exposant, pas la distance": 0.327,
          "n'oublie pas le signe moins dans l'exposant": 12.1825})
wb.record("13.5g", math.exp(-1.5 * float((X_135 - Z_135) @ (X_135 - Z_135))) * 1e5, decimals=2, mistakes={"multiplie le résultat par 100 000": 0.0})
wb.record("13.5h", 1000 + 1000 * 999 // 2, mistakes={"compte chaque paire j < l une seule fois": 1_000_000,
          "ajoute les 1 000 carrés x_j²": 499_500,
          "il n'y a pas de paire j = l parmi les produits croisés": 501_500})

WB_ANSWER {"decimals": 2, "hash": "7d6b1e56522a2d48a384b27a7819155f339023340c5feeeedb322f4022d4ea14", "hash_coarse": "f61db8f3cae432458567d7880542eaddb02b29861cc1ee4cf023d8c3eb08ddd7", "id": "13.5a", "kind": "float", "magnitude": 0, "mistakes": {"04babd68f7d454be3db0c9a3ea9b26d6cf3f5ce655ce96832cfc78515ebed4ac": "multiplie les coordonnées de même rang : la première avec la première, la seconde avec la seconde"}, "sign": 1}
📝 Ex 13.5a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "9bf607f0088f9bcafe7f5e93873e5b29c01e2e154458927b32bfc9049ab4558d", "hash_coarse": "bd650806f2affc7525da7bcb2a382cfaa14d035bdbbb6d7db6764cd0fdd2471f", "id": "13.5b", "kind": "float", "magnitude": 1, "mistakes": {"6a378c73ca461b14614b31e50165d787aca5fb4cd7a2487d1bae7766ba386bc1": "relis la définition du noyau polynomial : où est le degré 2 ?"}, "sign": 1}
📝 Ex 13.5b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "8531dbf5d2f98ae7eb072e480e6f6e89e

**Ex 13.6 — Anglais ou français ? Naïve Bayes multinomial sur des lettres**

In [23]:
wb.record("13.6a", 0.0, decimals=2, mistakes={"sans lissage, compte de w en français divisé par le total français : combien de w en français ?": 0.02})
wb.record("13.6b", float(TH_EN[3]), decimals=3, mistakes={"c'est la valeur sans lissage": 0.14,
          "le total gagne α p = 4, pas 1": 0.157,
          "le compte gagne aussi α = 1": 0.13})
wb.record("13.6c", float(TH_FR[3]), decimals=4, mistakes={"la question est posée avec lissage : le compte gagne-t-il α ?": 0, "le total gagne α p = 4, pas 1": 0.0196})
wb.record("13.6d", float(TH_FR[0]), decimals=4, mistakes={"c'est la valeur sans lissage": 0.76, "le total gagne α p = 4, pas 1": 0.7647})
wb.record("13.6e", float(L_EN), decimals=3, mistakes={"multiplie log θ_e par le compte de e, 2": -4.399,
          "n'oublie pas le log du prior": -4.443,
          "les logarithmes sont népériens": -2.152})
wb.record("13.6f", float(L_FR), decimals=3, mistakes={"multiplie log θ_e par le compte de e, 2": -7.61,
          "n'oublie pas le log du prior": -7.019,
          "les logarithmes sont népériens": -3.447})
wb.record("13.6g", posterior_first(L_EN, L_FR), decimals=3, mistakes={"tu as le posterior de l'autre langue": 0.048,
          "normalise les probabilités, pas les log-probabilités": 0.384})
wb.record("13.6h", 0.0, decimals=2, mistakes={"quelle langue la question vise-t-elle ? Reprends ta réponse a) dans la vraisemblance de cette langue": 1.0,
          "la question est posée sans lissage : que vaut P(w | FR) ?": 0.05})
wb.record("13.6i", posterior_first(math.log(0.6) + QUE @ np.log(TH_EN), math.log(0.4) + QUE @ np.log(TH_FR)), decimals=3, mistakes={"tu as le posterior de l'autre langue": 0.791,
          "n'oublie pas les priors 0,6 et 0,4": 0.15})

WB_ANSWER {"decimals": 2, "hash": "d88ffdcd5d07dbaa52518b244374544fc9b49ecabdf104960af62b186ec72a74", "hash_coarse": "04cbd73d9174e69bbce2d2452c9127b9a380f344f57c7b343724ee3a75a7013e", "id": "13.6a", "kind": "float", "magnitude": null, "mistakes": {"1e12246b72b5db99691ab682ab3cbdc0ba517a6d5dbba3f3dc5246c959a22a98": "sans lissage, compte de w en français divisé par le total français : combien de w en français ?"}, "sign": 0}
📝 Ex 13.6a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "f2796733f9ec4e9eb1606403e384cf57983cbc48638fa42dd53021410c696dad", "hash_coarse": "81ff58efddd1f880f44362902138db0d40a57270dd7c77241cd8530f141d1599", "id": "13.6b", "kind": "float", "magnitude": -1, "mistakes": {"b49b320ce067d3edd3831fbcb458f6462bd61c60add9727484de5ac38b107e15": "c'est la valeur sans lissage", "b70b5f39c59fb56f811788bfb6b37ad785fa98d6ced0a216bc67e7a727c13f8e": "le compte gagne aussi α = 1", "bd25bc89cd1b11afb29f95e4b353ccb56515083e36e1a677f2f97beab6540823": "

**Ex 13.7 — Posterior d'un Naïve Bayes gaussien**

In [24]:
wb.record("13.7a", float(G_137.mean()), decimals=2, mistakes={})
wb.record("13.7b", float(G_137.var()), decimals=2, mistakes={"Naïve Bayes divise par n, pas par n − 1": 41.67, "c'est l'écart-type ; on demande la variance": 5.59})
wb.record("13.7c", float(LA_FL), decimals=3, mistakes={"dans 2πσ², σ² est la variance 6,5², pas l'écart-type": -4.518,
          "n'oublie pas le facteur ½ devant le logarithme": -8.244,
          "n'oublie pas le terme −½ log(2πσ²)": -2.663})
wb.record("13.7d", float(LP_FL), decimals=3, mistakes={"dans 2πσ², σ² est la variance 6,5², pas l'écart-type": -3.559,
          "n'oublie pas le facteur ½ devant le logarithme": -7.286,
          "n'oublie pas le terme −½ log(2πσ²)": -1.704})
wb.record("13.7e", float(LA_BD), decimals=3, mistakes={"dans 2πσ², σ² est la variance 1,2², pas l'écart-type": -2.135,
          "n'oublie pas le terme −½ log(2πσ²)": -1.125})
wb.record("13.7f", float(LP_BD), decimals=3, mistakes={"n'oublie pas le facteur ½ devant le logarithme": -2.963,
          "n'oublie pas le terme −½ log(2π)": -1.125})
wb.record("13.7g", float(JA_137), decimals=3, mistakes={"n'oublie pas le log du prior": -7.68,
          "additionne le log du prior, ne le multiplie pas": 4.591})
wb.record("13.7h", float(JP_137), decimals=3, mistakes={"n'oublie pas le log du prior": -6.539,
          "additionne le log du prior, ne le multiplie pas": 5.221})
wb.record("13.7i", posterior_first(JP_137, JA_137), decimals=3, mistakes={"tu as le posterior de l'autre espèce": 0.281,
          "n'oublie pas les priors 0,55 et 0,45": 0.758})
wb.record("13.7j", posterior_first(math.log(0.45) + LP_FL, math.log(0.55) + LA_FL), decimals=3, mistakes={"tu as le posterior de l'autre espèce": 0.319,
          "n'oublie pas les priors 0,55 et 0,45": 0.723})
wb.record("13.7k", False, mistakes={"que suppose le produit des deux densités ? Relis l'encadré sur l'indépendance conditionnelle (§13.6)": True})

WB_ANSWER {"decimals": 2, "hash": "99909aa2362b087c30dca46be3eb6b681602d1054440d6503246ff7042492500", "hash_coarse": "740ef33c9013ca1cbcd14cafbf4275ffab93055e8a10e07f1e59838d8d8a5cc1", "id": "13.7a", "kind": "float", "magnitude": 2, "sign": 1}
📝 Ex 13.7a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"alt_coarse": ["b9dcfadd3c6b0422dc491eca95d71c81ea62410eb4d83bec9e5b2746a698bfe5"], "decimals": 2, "hash": "e218e081d4a876c79a9d5c464af1a8e737144b6a3919f88ccbd2c9644a6c2f77", "hash_coarse": "e3765170a54d84e955052485dd20f527986bf831bee720f9a632b48fcf0cd574", "id": "13.7b", "kind": "float", "magnitude": 1, "mistakes": {"6a5c8418fb44b8ca9f2f12ccf6174ccbede145868dd02446294659b60cee10da": "c'est l'écart-type ; on demande la variance", "fea8b78341ca591fdafbe6b653191d494bbd9f4b17c2f23c422f3b91cce83343": "Naïve Bayes divise par n, pas par n − 1"}, "sign": 1}
📝 Ex 13.7b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "c5189216e83801e8e44a14058890eec9f86cf93

**Ex 13.9 — Mémoire et temps d'un kNN sur MNIST complet**

In [25]:
wb.record("13.9a", N_TRAIN * P_PIX, mistakes={"le kNN garde l'entraînement, pas le test": 7_840_000,
          "tu as compté l'entraînement et le test ; le kNN ne garde que l'entraînement": 54_880_000})
wb.record("13.9b", N_TRAIN * P_PIX * 8 / 1e6, decimals=2, mistakes={"l'énoncé définit le Mo comme 10⁶ octets, pas 2²⁰": 358.89,
          "combien d'octets occupe un float64 ?": 188.16, "combien d'octets occupe un float64 ? (pas un seul)": 47.04})
wb.record("13.9c", N_TRAIN * P_PIX / 1e6, decimals=2, mistakes={"l'énoncé définit le Mo comme 10⁶ octets, pas 2²⁰": 44.86, "combien d'octets occupe un uint8 ?": 376.32})
wb.record("13.9d", 3 * N_TRAIN * P_PIX, mistakes={"combien d'opérations par valeur ?": 47_040_000})
wb.record("13.9e", 3 * N_TRAIN * P_PIX * N_TEST / 1e10, decimals=2, mistakes={"c'est le temps d'une seule requête ; il y en a 10 000": 0.01,
          "combien d'opérations par valeur ?": 47.04})
wb.record("13.9f", N_TEST * N_TRAIN * 8 / 1e9, decimals=2, mistakes={"combien d'octets occupe un float64 ?": 0.6, "en quelle unité la question demande-t-elle la taille ?": 4800})
wb.record("13.9g", 500 * N_TRAIN * 8 / 1e6, decimals=2, mistakes={"combien d'octets occupe un float64 ?": 30, "un batch compte combien d'images de test ?": 4800})
wb.record("13.9h", 10 * (P_PIX + 1), mistakes={"n'oublie pas les 10 biais": 7840, "un jeu de poids par classe : 10 classes": 785})
wb.record("13.9i", round(N_TRAIN * P_PIX * 8 / (10 * (P_PIX + 1) * 8)), mistakes={"n'oublie pas les 10 biais dans la régression logistique": 6000,
          "compare des octets à des octets : 8 octets par paramètre aussi": 47939})
wb.record("13.9j", N_TRAIN * 50 * 8 / 1e6, decimals=2, mistakes={"combien d'octets occupe un float64 ?": 3, "combien de valeurs garde chaque image après la PCA ?": 376.32})

WB_ANSWER {"hash": "af62ef39bc8963257d971b41fdd914fcea94a74260b0f1dc53d5c5bf37c71899", "id": "13.9a", "kind": "int", "magnitude": 7, "mistakes": {"9f7bd6e9615e34312d0afc1a9d863c3be8730879f6fe5fbddd5373a9d669889d": "le kNN garde l'entraînement, pas le test", "f0630d21ea6b6feb6229a04dfc443c4edecd7cff58d746d95c7baabc14cf5f36": "tu as compté l'entraînement et le test ; le kNN ne garde que l'entraînement"}, "sign": 1}
📝 Ex 13.9a : réponse enregistrée (int).
WB_ANSWER {"decimals": 2, "hash": "93f3d81520f0f9c6711206a142814ff30429cf55cdd80c63831cc10c0f9c9969", "hash_coarse": "2fa7aa7c86bd7719ca14cbc00b7b90252aa22fceb3f85b16b092de8c77364efc", "id": "13.9b", "kind": "float", "magnitude": 2, "mistakes": {"38a567907f35f4041d59dcaa72150c54670a2154ac8d4fa8ae9e4bc7b2ad224f": "combien d'octets occupe un float64 ?", "6437db48cd0cd01771d69bf882666bebd8971df363763be5bb81bd6889799496": "l'énoncé définit le Mo comme 10⁶ octets, pas 2²⁰", "cbec0db7dc05d3efb0af3689975333b001abb66ffad4212d5a42c5b498502722": "

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire, pour chaque classifieur du chapitre, ce qu'il garde en mémoire, où il passe son temps et ce qu'il suppose des données ?
2. Sais-tu calculer à la main un vote kNN, une marge, un gain de split et un posterior de Naïve Bayes ?
3. Sais-tu choisir une baseline et régler ses hyperparamètres par validation croisée ?

**Pour aller plus loin** : les guides utilisateur de scikit-learn sur les plus proches voisins, les SVM, les arbres et Naïve Bayes, et l'article de Domingos et Pazzani, cités dans la fiche. La suite : le ch. 14 (ensembles), qui construira forêts aléatoires, AdaBoost et gradient boosting avec tes arbres de `mylearn.tree`.

*Les exercices de code 13.12 à 13.36 (parties A à F, ta librairie `neighbors`, `tree`, `naive_bayes` et `logistic`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 13` pour obtenir le notebook complet.*